# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yelsebety/Yassin-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking, built from a score.** My lane (Lane 2, Refresh / Content Opportunity Scoring) asks *"which page should a reviewer look at first?"* The answer is an ordered list, not a yes/no for every page. A classifier (predicting the chance a page is declining) is the tool underneath, but the output I care about is the order of the list.

It is not clustering (I have a target to aim at, not just groups to discover).

**The loop:** observable page signals → a score per page → a ranked review queue → a reviewer takes an action (refresh, expand, protect, prune, or monitor) → later, we check if the top of the list was right.

In [11]:
import os, sys, subprocess
import numpy as np
import pandas as pd

# Find the repo root so the CSV loads in Colab and locally.
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/yelsebety/Yassin-Flyrank.git"
REPO_DIR = "Yassin-Flyrank"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

raw = pd.read_csv("data/raw/content_refresh_anonymized.csv")

loop = pd.DataFrame(
    [
        ("Input", "observable page signals (impressions, position, CTR, age, freshness, content type...)"),
        ("Model / score", "probability that a page is declining, used as a priority score"),
        ("Output", "ranked review queue with reason codes"),
        ("Action", "reviewer chooses: refresh, expand, protect, prune, or monitor"),
        ("Check", "precision@K on the top of the list vs. a simple rule baseline"),
    ],
    columns=["Step in the ML loop", "In my lane"],
)
loop

,Step in the ML loop,In my lane
0,Input,"observable page signals (impressions, position..."
1,Model / score,"probability that a page is declining, used as ..."
2,Output,ranked review queue with reason codes
3,Action,"reviewer chooses: refresh, expand, protect, pr..."
4,Check,precision@K on the top of the list vs. a simpl...


## 2. Target or proxy

**What I would predict:** whether a page is *declining* (losing visibility).

**Where the label comes from (starter data):** a defined rule, so it is a **proxy**, not a real outcome:

`is_declining_label = (trend_direction == "down")`

and `trend_direction` is computed from the last 30 days of impressions vs. the 30 days before. "Down" means a drop of more than 20%.

**Why that matters:** the label comes from a rule on the *same window* the features are measured in, so a model could just learn the rule. That is why `trend_direction` and `trend_pct` are never features.

**Better target for the capstone:** an observed *future* outcome, such as features from the prior 90 days → did impressions drop over the *next* 30 days? I would build that from the warehouse daily table, with a leakage check.

In [12]:
# Sketch of the target column (starter proxy label)
raw["is_declining_label"] = (raw["trend_direction"] == "down").astype(int)

cols = ["content_id", "impressions_prev_30d", "impressions_last_30d", "trend_pct", "trend_direction", "is_declining_label"]
display(raw[raw["trend_direction"].isin(["down", "stable", "up"])][cols].sample(8, random_state=0))

# Prove the label really is the 30-day comparison (a rule, not an observed future outcome)
down = raw[raw["is_declining_label"] == 1]
assert (down["impressions_last_30d"] < 0.8 * down["impressions_prev_30d"]).all()
print(f"Label rate: {raw['is_declining_label'].mean():.1%} of pages are 'down' ({raw['is_declining_label'].sum():,} of {len(raw):,}).")
print("Label = last-30d vs prev-30d impressions, so trend_direction / trend_pct can never be features.")

,content_id,impressions_prev_30d,impressions_last_30d,trend_pct,trend_direction,is_declining_label
18137,content_1cfcbdddefaa,408,550,34.8,up,0
21688,content_51dbefcd6b41,6744,6114,-9.3,stable,0
24945,content_eb79a33071e7,18,11,-38.9,down,1
12010,content_ec36d29c9ca1,36,144,300.0,up,0
10200,content_eca3088b610c,1,3,200.0,up,0
11917,content_24b23341e0be,775,637,-17.8,stable,0
9594,content_8719691f4dce,2330,806,-65.4,down,1
19043,content_48cecb04b88e,8,3,-62.5,down,1


Label rate: 54.2% of pages are 'down' (16,262 of 30,000).
Label = last-30d vs prev-30d impressions, so trend_direction / trend_pct can never be features.


## 3. Success metric

**Metric: precision@K**, with K = 50 (and 20 as a stricter check). Of the top K pages the system tells a reviewer to look at first, how many really are positive?

**Why this one:** a reviewer only has time for a short list, so what matters is how good the *top* of the list is. Overall accuracy would hide that.

**What "good" means:** precision@50 clearly above both the base rate (about 54% of pages are "down") and a simple rule baseline, measured with a client-holdout split (whole clients kept out of training). Secondary checks: average precision and recall.

**Caveat:** on the starter data this measures the *proxy* label. It is a way to practise the method, not proof that pages will really decline.

In [13]:
def precision_at_k(scores, labels, k):
    """Share of the top-k scored pages that are positive."""
    top = scores.sort_values(ascending=False).head(k).index
    return labels.loc[top].mean()

base_rate = raw["is_declining_label"].mean()

# A naive baseline I can compute today: "most visible pages first"
print(f"Base rate (random list would get about): {base_rate:.3f}")
for k in (20, 50):
    p = precision_at_k(raw["impressions_90d"], raw["is_declining_label"], k)
    print(f"Naive baseline 'most impressions first'  precision@{k}: {p:.3f}")
print("A real score has to beat both numbers on held-out clients to count as useful.")

Base rate (random list would get about): 0.542
Naive baseline 'most impressions first'  precision@20: 0.450
Naive baseline 'most impressions first'  precision@50: 0.420
A real score has to beat both numbers on held-out clients to count as useful.


## 4. The unit of analysis, as a real dataframe

**One row = one content item (a page), described by its trailing-90-day search and engagement numbers.** The key is `content_id`, and `client_id` is only for grouping and holdout splits.

**My lane's slice:** pages with at least 1 impression and at least 90 days of age (the same filter the starter pipeline uses). Columns are grouped by role below.

In [14]:
slice_ = raw[(raw["impressions_90d"] > 0) & (raw["content_age_days"] >= 90)].copy()

id_cols      = ["content_id", "client_id"]                       # grouping only, never features
feature_cols = ["impressions_90d", "avg_position", "ctr", "content_age_days",
                "days_since_last_update", "content_type"]        # a few example signals
target_col   = ["is_declining_label"]                            # proxy target (section 2)

lane_df = slice_[id_cols + feature_cols + target_col]
print(f"Lane slice: {lane_df.shape[0]:,} rows x {lane_df.shape[1]} columns")
assert lane_df["content_id"].is_unique, "one row should be one content item"
print("Unit check: content_id is unique -> one row = one page.")
lane_df.head(8)

Lane slice: 30,000 rows x 9 columns
Unit check: content_id is unique -> one row = one page.


,content_id,client_id,impressions_90d,avg_position,ctr,content_age_days,days_since_last_update,content_type,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,10.6,0.76,187,20,keyword article,1
1,content_a1fb4e703a9e,client_4e07408562,15320,20.3,0.05,445,25,keyword article,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,36.5,0.09,141,20,keyword article,1
3,content_331d6c4de07b,client_19581e27de,11751,6.2,0.49,463,22,keyword article,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,44.0,0.13,263,14,keyword article,1
5,content_d4084a4bc775,client_f369cb89fc,3970,8.5,0.03,147,20,keyword article,1
6,content_9a34b442b552,client_8722616204,20,7.0,0.00,90,20,keyword article,1
7,content_a63219c6e95a,client_19581e27de,1724,21.2,0.06,445,22,keyword article,0


## 5. Why ML beats a fixed rule here

**What a fixed rule does:** each if-statement catches one slice of the problem. In the table below, the very precise rule flags only 17 pages (it finds almost none of the declining pages), while the broad rules flag thousands but their precision stays close to the base rate (one is slightly below it, the best is about 63%).

**Why the pattern is messy:** how likely a page is to be "down" depends on several signals at once, for example content type and position tier (also below), and those signals interact. Writing and tuning all those thresholds by hand does not scale across 32 clients.

**Evidence from the starter pipeline:** its committed report (`outputs/model_report.md`) shows precision@50 of 0.240 for the rule baseline and 0.740 for the best model (random forest, client-holdout split). That *suggests* learning can beat a fixed rule on this slice, but it is on the proxy label, so I would have to earn it again with a future-window target.

**Honest limit:** if a simple rule turns out to be just as good, that is a valid finding and I would say so.

In [15]:
d = slice_
y = d["is_declining_label"]

rules = {
    "stale_visible: 180+ days since update, 500+ impressions": (d["days_since_last_update"] >= 180) & (d["impressions_90d"] >= 500),
    "thin_visible: under 1200 words, 250+ impressions": (d["word_count"] > 0) & (d["word_count"] < 1200) & (d["impressions_90d"] >= 250),
    "page_one_decay: position <= 10, age 180+ days": (d["avg_position"] > 0) & (d["avg_position"] <= 10) & (d["content_age_days"] >= 180),
    "low_ctr_visible: 500+ impressions, position <= 20, CTR < 0.5%": (d["impressions_90d"] >= 500) & (d["avg_position"] > 0) & (d["avg_position"] <= 20) & (d["ctr"] < 0.5),
}
rows = [(name, int(m.sum()), y[m].mean(), y[m].sum() / y.sum()) for name, m in rules.items()]
rule_table = pd.DataFrame(rows, columns=["single rule", "pages flagged", "precision (share 'down')", "recall (share of all 'down' found)"])
print(f"Base rate: {y.mean():.3f}")
display(rule_table.round(3))

print("Decline rate by content type and position tier (one signal at a time):")
display(d.groupby("content_type")["is_declining_label"].agg(["mean", "count"]).round(3))
display(d.groupby("position_tier")["is_declining_label"].agg(["mean", "count"]).round(3))

Base rate: 0.542


,single rule,pages flagged,precision (share 'down'),recall (share of all 'down' found)
0,"stale_visible: 180+ days since update, 500+ im...",17,0.941,0.001
1,"thin_visible: under 1200 words, 250+ impressions",82,0.537,0.003
2,"page_one_decay: position <= 10, age 180+ days",7076,0.518,0.225
3,"low_ctr_visible: 500+ impressions, position <=...",9759,0.627,0.376


Decline rate by content type and position tier (one signal at a time):


,mean,count
content_type,,
comparison article,0.572,697
feedly article,0.287,2096
keyword article,0.561,27207


,mean,count
position_tier,,
deep,0.344,1319
page_1,0.570,11814
page_3_5,0.562,7242
striking,0.610,7304
top_3,0.241,2321


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.